In [ ]:
import pypsa
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from _helpers_notebooks import mock_snakemake

snakemake = mock_snakemake(
    "plot_compare_lcox",
    scenario="default",  # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
    wacc="uniform",  # uniform or regional
    cost_year="2050",
)

In [ ]:
config = snakemake.config

In [ ]:
add_iron_ore_cost = True

In [ ]:
colors = config["colors"]

### Read networks in Network collection

In [ ]:
scenarios = snakemake.input.keys()

In [ ]:
nc = {}

for scenario in scenarios:
    model_fn = snakemake.input[scenario]

    n = pypsa.Network(model_fn)
    n.name = scenario
    nc[scenario] = n

nc = pypsa.NetworkCollection(list(nc.values()))

### Adjust carrier name before statistics

In [ ]:
# Rename for all scenarios
for scenario in scenarios:
    nc[scenario].links["carrier"] = nc[scenario].links["carrier"].replace({
        "battery_elec": "battery discharge",
        "renewable_electricity": "battery charge"
    })

### Costs

In [ ]:
hbi_demand = nc.statistics.withdrawal().to_frame().loc["Load", :, "hbi_demand"]
# hbi_demand

In [ ]:
df = nc.statistics.system_cost(groupby="carrier").to_frame("systemcost")
# df

In [ ]:
fom = nc.statistics.fom(groupby="carrier").to_frame("fom")

In [ ]:
df["systemcost+fom"] = df["systemcost"].add(fom["fom"], fill_value=0)

In [ ]:
df["systemcost+fom_per_hbi"] = df["systemcost+fom"].div(hbi_demand.squeeze().rename_axis("network"), level="network")
# df

### Iron ore

In [ ]:
iron_ore_cost = config["iron_ore"]["marginal_cost"] * config["iron_ore"]["ore_to_steel_ratio"] 

### Plot

In [ ]:
# Pivot to get networks as rows and carriers as columns
plot_df = df.pivot_table(
    index="network", columns="carrier", values="systemcost+fom_per_hbi", aggfunc="sum"
)

if add_iron_ore_cost:
    plot_df["iron ore"] = iron_ore_cost
else: 
    plot_df["iron ore"] = 0

plot_df.rename(
    columns={
        "renewable_solar": "solar",
        "renewable_onwind": "onshore wind",
        "battery charge": "battery inverter (charging)",
        "battery discharge": "battery inverter (discharging)",
        "battery_elec": "battery",
        "hydrogen": "hydrogen storage",
        "electrolysis": "electrolysis",
        "direct_reduction_furnace": "direct reduction furnace",
    },
    inplace=True,
)

order = ["iron ore", "solar", "onshore wind", "battery inverter (charging)", "battery inverter (discharging)", "battery", "hydrogen storage", "electrolysis", "direct reduction furnace"]

plot_df = plot_df[order]

In [ ]:

# --- Parse scenario names into region and quantity ---
qty_map = {"01": 0.1, "1": 1.0, "10": 10.0, "100": 100.0}
region_map = {"europe": "Europe", "south_america": "South America"}

def parse_scenario(name):
    region_raw, qty_raw = name.rsplit("_", 1)
    region = region_map.get(region_raw, region_raw.replace("_", " ").title())
    qty = qty_map.get(qty_raw, float(qty_raw))
    return region, qty

plot_df["region"] = [parse_scenario(s)[0] for s in plot_df.index]
plot_df["quantity"] = [parse_scenario(s)[1] for s in plot_df.index]

# South America on the left, Europe on the right
groups_ordered = ["South America", "Europe"]
plot_df["region_order"] = plot_df["region"].map({g: i for i, g in enumerate(groups_ordered)})
plot_df = plot_df.sort_values(["region_order", "quantity"]).drop(columns="region_order")

# --- Compute x positions with a gap between region groups ---
bar_width = 0.5
group_gap = 1

x_positions = []
group_centers = {}
group_spans = {}
current_x = 0

for g in groups_ordered:
    mask = plot_df["region"] == g
    n = int(mask.sum())
    positions = list(range(current_x, current_x + n))
    x_positions.extend(positions)
    group_centers[g] = np.mean(positions)
    group_spans[g] = (positions[0], positions[-1])
    current_x += n + group_gap

# --- Create stacked bar plot ---
fig, ax = plt.subplots(figsize=(10, 4.5))

bottom = np.zeros(len(plot_df))
for col in order:
    if col in plot_df.columns:
        values = plot_df[col].fillna(0).values
        ax.bar(x_positions, values, bottom=bottom, width=bar_width,
               label=col, color=config["colors"][col], alpha=0.65)
        bottom += values

# --- Totals on top of bars ---
totals = plot_df[order].sum(axis=1).values
for x, total in zip(x_positions, totals):
    ax.text(x, total, f"{total:.1f}", ha="center", va="bottom", fontsize=9)

# --- Primary axis formatting ---
ax.set_ylabel("Cost of HBI in €/t$_{hbi}$")
ax.set_xlabel("")
ax.yaxis.grid(True, linestyle="-", alpha=0.6)
ax.set_axisbelow(True)

# --- Inner x-axis: quantity labels ---
ax.set_xticks(x_positions)
qty_labels = [f"{row['quantity']:.4g} Mt" for _, row in plot_df.iterrows()]
ax.set_xticklabels(qty_labels, rotation=0, fontsize=9)
ax.tick_params(axis="x", length=0)

# --- Outer x-axis: region group labels with bracket (moved up) ---
xaxis_transform = ax.get_xaxis_transform()
y_bracket = -0.11
y_label_pts = -7

for g in groups_ordered:
    center = group_centers[g]
    x_start, x_end = group_spans[g]

    ax.annotate(
        "",
        xy=(x_end + bar_width / 2 + 0.05, y_bracket),
        xycoords=xaxis_transform,
        xytext=(x_start - bar_width / 2 - 0.05, y_bracket),
        textcoords=xaxis_transform,
        arrowprops=dict(arrowstyle="-", color="black", lw=0.8),
        annotation_clip=False,
    )
    ax.annotate(
        g,
        xy=(center, y_bracket),
        xycoords=xaxis_transform,
        xytext=(0, y_label_pts),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=10,
        annotation_clip=False,
    )

# --- % difference annotation: leftmost Europe vs leftmost South America ---
plot_df_idx_list = list(plot_df.index)
eu_first = plot_df[plot_df["region"] == "Europe"].index[0]
sa_first = plot_df[plot_df["region"] == "South America"].index[0]

eu_first_xpos = x_positions[plot_df_idx_list.index(eu_first)]
sa_first_xpos = x_positions[plot_df_idx_list.index(sa_first)]

totals_series = plot_df[order].sum(axis=1)
y_eu_top = float(totals_series[eu_first])
y_sa_top = float(totals_series[sa_first])

# Europe is the base: negative means SA is cheaper
pct_diff = (y_sa_top - y_eu_top) / y_eu_top * 100

# Red horizontal line from Europe's leftmost bar top across to directly above SA's leftmost bar
ax.plot([sa_first_xpos, eu_first_xpos], [y_eu_top, y_eu_top],
        color="red", linewidth=0.8, clip_on=False, zorder=5)

# Downward arrow and percentage annotation
ax.annotate("", xy=(sa_first_xpos, y_sa_top+20),
    xytext=(sa_first_xpos, y_eu_top),
    arrowprops=dict(arrowstyle="-|>", color="red", lw=0.8),
    annotation_clip=False,
)

ax.text(
    sa_first_xpos - 0.1,
    (y_eu_top + y_sa_top) / 2,
    f"{pct_diff:.1f}%",
    ha="right",
    va="center",
    fontsize=8,
    color="red",
)

# --- Legend ---
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    title="Cost component",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()
plt.savefig(snakemake.output.lcox_comparison, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.lcox_comparison_png, dpi=300, bbox_inches="tight")
plt.show()

<!-- ### Statistics clarifiction (to be deleted) -->

In [ ]:
# nc["europe_100"].objective / 1e6

In [ ]:
# system_cost = nc["europe_100"].statistics.system_cost().div(1e6).sum()
# system_cost.round(2)

In [ ]:
# fom = nc["europe_100"].statistics.fom().div(1e6).sum()
# fom.round(2)

In [ ]:
# expanded_capex = nc["eu_1"].statistics.expanded_capex().div(1e6).sum()
# expanded_capex.round(2)

In [ ]:
# opex = nc["eu_1"].statistics.opex().div(1e6).sum()
# opex.round(2)

<!-- -> system_cost = expanded_capex + opex -->

In [ ]:
# objective = nc["eu_1"].objective / 1e6
# objective.round(2)

<!-- -> objective = system_cost + fom = expanded_capex + opex + fom -->

In [ ]:
# nc["eu_1"].statistics.overnight_cost().div(1e6).sum()